# 06 · RQ1: How often do the two models disagree?

Report section 3.2: Table 5, Table 6, Figure 4 and all numbers in the text.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    P = Path("/content/drive/MyDrive/Affective Project/predictions")
else:
    P = Path("../predictions")

FILES = {
    "audio_v2_test": P / "audio_v2_test_predictions.csv",
    "audio_v2_validation": P / "audio_v2_validation_predictions.csv",
    "audio_v1_validation": P / "audio_v1_validation_predictions.csv",
    "audio_v2_history": P / "audio_v2_history.csv",
    "visual_test": P / "visual_test_predictions.csv",
    "visual_validation": P / "visual_validation_predictions.csv",
}
missing = [str(p) for p in FILES.values() if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(missing)
for k, p in FILES.items():
    print(f"{k:22s} {p}")

Mounted at /content/drive
audio_v2_test          /content/drive/MyDrive/Affective Project/predictions/audio_v2_test_predictions.csv
audio_v2_validation    /content/drive/MyDrive/Affective Project/predictions/audio_v2_validation_predictions.csv
audio_v1_validation    /content/drive/MyDrive/Affective Project/predictions/audio_v1_validation_predictions.csv
audio_v2_history       /content/drive/MyDrive/Affective Project/predictions/audio_v2_history.csv
visual_test            /content/drive/MyDrive/Affective Project/predictions/visual_test_predictions.csv
visual_validation      /content/drive/MyDrive/Affective Project/predictions/visual_validation_predictions.csv


In [2]:
D = Path("/content/drive/MyDrive/Affective Project/data") if IN_COLAB else Path("../data")
DATA = {"ratings": D / "summaryTable.csv", "actors": D / "VideoDemographics.csv"}
missing = [str(p) for p in DATA.values() if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(missing)

In [3]:
# Helper functions used below
CLASSES = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad"]
PROB = [f"prob_{c}" for c in CLASSES]


def load_split(split):
    """One row per clip: true label and both models' predictions and probabilities.
    Clips without a visual prediction are dropped (only 1076_MTI_SAD_XX in validation)."""
    v = pd.read_csv(FILES[f"visual_{split}"])
    v = v[v["status"] == "ok"]
    a = pd.read_csv(FILES[f"audio_v2_{split}"])
    a["video_id"] = a["filename"].str.replace(".wav", "", regex=False)
    m = v[["video_id", "Speaker", "true_label", "predicted_label", *PROB]].merge(
        a[["video_id", "true_label", "predicted_label", *PROB]],
        on="video_id", suffixes=("_v", "_a"), validate="one_to_one")
    assert (m["true_label_v"] == m["true_label_a"]).all(), "label mismatch"
    m = m.rename(columns={"true_label_v": "y", "predicted_label_v": "pred_v",
                          "predicted_label_a": "pred_a"}).drop(columns="true_label_a")
    return m


def ece(p, y_idx, bins=10):
    """Expected calibration error with 10 equal-width confidence bins."""
    conf, pred = p.max(1), p.argmax(1)
    correct = (pred == y_idx).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.any():
            total += s.mean() * abs(conf[s].mean() - correct[s].mean())
    return total

In [4]:
CODE = {"A": "Anger", "D": "Disgust", "F": "Fear", "H": "Happy", "N": "Neutral", "S": "Sad"}


def add_human_ratings(df):
    s = pd.read_csv(DATA["ratings"], dtype=str).rename(columns={"FileName": "video_id"})
    df = df.merge(s[["video_id", "VoiceVote", "FaceVote", "MultiModalVote"]],
                  on="video_id", how="left", validate="one_to_one")
    assert df.VoiceVote.notna().all()
    for col, new in [("VoiceVote", "h_voice"), ("FaceVote", "h_face"), ("MultiModalVote", "h_av")]:
        df[new] = df[col].map(lambda x: frozenset(CODE[c] for c in x.split(":")))
        df[new + "_correct"] = [y in s for y, s in zip(df.y, df[new])]
    df["human_disagree"] = [len(a & b) == 0 for a, b in zip(df.h_voice, df.h_face)]
    return df

In [5]:
from scipy.stats import chi2_contingency

test = load_split("test")
test["correct_v"] = test.pred_v == test.y
test["correct_a"] = test.pred_a == test.y
test["disagree"] = test.pred_v != test.pred_a
test["intensity"] = test.video_id.str.split("_").str[3]
test = add_human_ratings(test)
print("Test clips:", len(test))

Test clips: 1552


## 3.2.1 Overall disagreement (Table 5)

In [6]:
n = len(test)
agree, dis = test[~test.disagree], test[test.disagree]
rows = [
    ("Models agree", len(agree)),
    ("  Both correct", int(agree.correct_v.sum())),
    ("  Both wrong (same label)", int((~agree.correct_v).sum())),
    ("Models disagree", len(dis)),
    ("  Only visual correct", int((dis.correct_v & ~dis.correct_a).sum())),
    ("  Only audio correct", int((~dis.correct_v & dis.correct_a).sum())),
    ("  Both wrong (different labels)", int((~dis.correct_v & ~dis.correct_a).sum())),
]
table5 = pd.DataFrame(rows, columns=["outcome", "clips"])
table5["share_%"] = (100 * table5.clips / n).round(1)
print(table5.to_string(index=False))
print(f"\nAgreements that were correct: {100 * agree.correct_v.mean():.1f}%")
for name, k in [("only the visual model", 4), ("only the audio model", 5), ("neither model", 6)]:
    print(f"Disagreements where {name} was right: {100 * rows[k][1] / len(dis):.1f}%")

                        outcome  clips  share_%
                   Models agree    757     48.8
                   Both correct    717     46.2
        Both wrong (same label)     40      2.6
                Models disagree    795     51.2
            Only visual correct    339     21.8
             Only audio correct    335     21.6
  Both wrong (different labels)    121      7.8

Agreements that were correct: 94.7%
Disagreements where only the visual model was right: 42.6%
Disagreements where only the audio model was right: 42.1%
Disagreements where neither model was right: 15.2%


## 3.2.2 Disagreement by emotion and intensity (Figure 4a)

In [7]:
by_emotion = pd.DataFrame({
    "model_disagreement_%": 100 * test.groupby("y").disagree.mean(),
    "human_disagreement_%": 100 * test.groupby("y").human_disagree.mean(),
}).loc[CLASSES].round(1)
by_emotion

,model_disagreement_%,human_disagreement_%
y,,
Anger,48.7,35.1
Disgust,50.9,69.1
Fear,52.8,47.5
Happy,36.2,59.2
Neutral,42.7,3.5
Sad,74.7,34.3


In [8]:
ieo = test[test.intensity.isin(["LO", "MD", "HI"])]
(ieo.groupby("intensity").disagree.agg(["size", "mean"])
    .loc[["LO", "MD", "HI"]].assign(mean=lambda d: (100 * d["mean"]).round(1))
    .rename(columns={"size": "clips", "mean": "disagreement_%"}))

,clips,disagreement_%
intensity,,
LO,95,72.6
MD,95,51.6
HI,95,44.2


## 3.2.3 Is model disagreement a real conflict between voice and face? (Figure 4b)

In [9]:
hd = test.human_disagree
print(f"Clips where voice-only and face-only raters chose different emotions: "
      f"{hd.sum()} ({100 * hd.mean():.1f}%)")
print(f"Model disagreement when humans disagree:  {100 * test[hd].disagree.mean():.1f}%")
print(f"Model disagreement when humans agree:     {100 * test[~hd].disagree.mean():.1f}%")
chi2, p, _, _ = chi2_contingency(pd.crosstab(test.disagree, hd))
print(f"Chi-square test: chi2 = {chi2:.2f}, p = {p:.4f}")

Clips where voice-only and face-only raters chose different emotions: 658 (42.4%)
Model disagreement when humans disagree:  56.2%
Model disagreement when humans agree:     47.5%
Chi-square test: chi2 = 11.12, p = 0.0009


In [10]:
d = test[test.disagree].copy()
d["mirrored"] = [a in hv and v in hf for a, v, hv, hf in zip(d.pred_a, d.pred_v, d.h_voice, d.h_face)]
d["category"] = np.where(~d.human_disagree, "Model error",
                 np.where(d.mirrored, "Same labels as humans", "Humans also disagree"))
print(f"Model disagreements: {len(d)}")
print(f"  on clips where humans also disagree: {d.human_disagree.sum()} ({100 * d.human_disagree.mean():.1f}%)")
print(f"  on clips where humans agree (model error): {(~d.human_disagree).sum()} "
      f"({100 * (~d.human_disagree).mean():.1f}%)")
real = d.human_disagree & d.mirrored
print(f"  humans disagree and each model gives the label of the raters of its own channel: "
      f"{real.sum()} ({100 * real.mean():.1f}%)")

Model disagreements: 795
  on clips where humans also disagree: 370 (46.5%)
  on clips where humans agree (model error): 425 (53.5%)
  humans disagree and each model gives the label of the raters of its own channel: 78 (9.8%)


In [11]:
cat = pd.crosstab(d.y, d.category).loc[CLASSES, ["Model error", "Humans also disagree", "Same labels as humans"]]
cat_pct = (100 * cat.div(cat.sum(axis=1), axis=0)).round(0)
cat_pct["n_disagreements"] = cat.sum(axis=1)
cat_pct

category,Model error,Humans also disagree,Same labels as humans,n_disagreements
y,,,,
Anger,58.0,36.0,6.0,129
Disgust,21.0,65.0,14.0,135
Fear,49.0,40.0,11.0,140
Happy,28.0,46.0,26.0,96
Neutral,96.0,3.0,1.0,97
Sad,67.0,28.0,5.0,198


## 3.2.4 Comparison with human accuracy (Table 6)

In [12]:
pd.DataFrame({
    "information": ["Audio only", "Video only", "Audio and video"],
    "humans_%": [100 * test.h_voice_correct.mean(), 100 * test.h_face_correct.mean(),
                 100 * test.h_av_correct.mean()],
    "our_model_%": [100 * test.correct_a.mean(), 100 * test.correct_v.mean(), np.nan],
}).round(1)

,information,humans_%,our_model_%
0,Audio only,47.9,67.8
1,Video only,69.5,68.0
2,Audio and video,76.5,NaN


The fusion row of Table 6 (81.8% vs. 76.5% for humans, McNemar test) needs the
fusion model, so it is computed in notebook `09_rq5_fusion.ipynb`.